In [1]:
%load_ext autoreload
%autoreload 2

In [3]:

import logging
import os
from crmprtd import setup_logging


setup_logging(None, '/workspaces/crmprtd/fern/all_stations_insert/fern_all_station_log.log', 'tongli1997@uvic.ca', 'DEBUG', 'fern')
log = logging.getLogger(__name__)


import sqlalchemy as sa
from sqlalchemy.orm import Session
from pycds import Variable
from crmprtd.infer import infer
import pickle
from crmprtd.more_itertools import tap, log_progress
from crmprtd.align import align
from crmprtd.insert import insert
from pprint import pprint
from crmprtd.constants import InsertStrategy

print(os.getcwd())
engine = sa.create_engine("postgresql://tongli1997@pg01.pcic.uvic.ca:5432/crmp", echo=False)
session = Session(engine)

# result = session.execute(sa.text("SELECT * FROM crmp.meta_network"))
# test = result.fetchall()

# print(test)

results2 = session.query(Variable).filter(Variable.network_id == 11)
test2 = results2.all()
# pprint(test2)


output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


# file_name = 'updated_rows_with_unit_output_Barren.pickle'
# file_name = 'full_record_station1-2_updated_rows_all_stations.pkl'
file_name = 'updated_10_rows_all_stations.pkl'
file_path = os.path.join(output_folder, file_name)

# Load the pickle file
with open(file_path, 'rb') as f:
    rows = pickle.load(f)

# pprint(rows)
rows_insert = rows[0:100]

is_diagnostic = False
insert_strategy = InsertStrategy.BULK
bulk_chunk_size=1000
sample_size=50
network = '_test'


infer(session, rows_insert, diagnostic = is_diagnostic)

# Filter the observations by time period, then align them.
log.info("Align + filter: start")
observations = list(
    # Note: filter(None, <collection>) removes falsy values from <collection>,
    # in this case possible None values returned by align.
    filter(
        None,
        tap(
            log_progress(
                (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 5000),
                "align progress: {count}",
                log.info,
            ),
            (
                align(session, row, is_diagnostic)
                for row in rows_insert
                # if start_date <= row.time <= end_date
            ),
        ),
    )
)
log.info("Align + filter: done")

log.info(f"Count of observations to process: {len(observations)}")
if is_diagnostic:
    for obs in observations:
        log.info(obs)
else:

    log.info("Insert: start")
    results = insert(
        session,
        observations,
        strategy=insert_strategy,
        bulk_chunk_size=bulk_chunk_size,
        sample_size=sample_size,
    )
    log.info("Insert: done")
    log.info("Data insertion results", extra={"results": results, "network": network})

# Add this after setting up logging
log.info("Logging system initialized")


ERROR! Session/line number was not unique in database. History logging moved to new session 3
/workspaces/crmprtd/fern/all_stations_insert
{"asctime": "2025-11-04 18:40:47,944", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = Blackhawk"}
{"asctime": "2025-11-04 18:40:47,959", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-04 18:40:47,979", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: find_or_create_matching_history_and_station ('FLNRO-FERN', 'Blackhawk', 55.078885, -120.352171, ('diagnostic', False)) -> <pycds.orm.tables.History object at 0xffff50f1e850>"}
{"asctime": "2025-11-04 18:40:47,980", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = Barren"}
{"asctime": "2025-11-04 18:40:47,991", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-04 18:40:

In [ ]:
rows[570:572]

[Row(time=datetime.datetime(2023, 8, 4, 15, 0), val=16.654, variable_name='TempC', unit='celsius', network_name='FLNRO-FERN', station_id='CaribouPass', lat=58.35718, lon=-129.54637),
 Row(time=datetime.datetime(2023, 8, 4, 16, 0), val=16.129, variable_name='TempC', unit='celsius', network_name='FLNRO-FERN', station_id='CaribouPass', lat=58.35718, lon=-129.54637)]

In [4]:
output_folder = '/workspaces/crmprtd/rows_output/'

# file_name = 'updated_rows_with_unit_output_Barren.pickle'
file_name = 'updated_rows_with_unit_output_Barren.pickle'
file_path = os.path.join(output_folder, file_name)

# Load the pickle file
with open(file_path, 'rb') as f:
    single_rows = pickle.load(f)

pprint(single_rows[0:10])

[Row(time=datetime.datetime(2021, 7, 9, 12, 0), val=nan, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 13, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 14, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 15, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 16, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 17, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barr